## Notebook Summary

This notebook runs a multi-model evaluation loop in two iterations and compares outcomes.

### What It Does
- Generates a challenging evaluation question using a dedicated `question_raiser` model.
- Runs multiple competitor models in parallel on the same question (first iteration).
- Uses a `judge` model to rank answers, assign scores, and provide per-model recommendations.
- Runs all competitor models again (second iteration), where each model gets:
  1. the original question
  2. its own previous answer
  3. the judge's recommendation for improvement
- Judges the second-iteration answers again and compares iteration 1 vs iteration 2.
- Produces a final leaderboard by each model's best score across both iterations.

### Caching
- Optional cache support (`USE_RESULT_CACHE`) stores question, per-iteration results, and per-iteration judge decisions.
- Benefit: lower API cost and faster reruns during development/testing.
- Tradeoff: cached data can become stale if prompts or configs change.


In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
# Basic settings
PRINT_QUESTION = True
PRINT_ANSWER = True
USE_RESULT_CACHE = False
# Cache benefit: saves API cost and run time by reusing prior question/results/decisions.
# Cache disadvantage: can return stale outputs and hide behavior changes after prompt/model updates.

# Model confi and generic function for getting the answer for a prompt
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

# Cache model outputs during iteration so re-runs do not call all providers every time.
# This keeps development/testing faster and avoids unnecessary API costs.
NOTEBOOK_DIR = Path("1_foundations/community_contributions/misi")
RESULTS_CACHE_PATH = (
    NOTEBOOK_DIR / "2_lab2_excercise.results_cache.json"
    if NOTEBOOK_DIR.exists()
    else Path("2_lab2_excercise.results_cache.json")
)


def load_cache(cache_path=RESULTS_CACHE_PATH):
    if not USE_RESULT_CACHE:
        return {}
    if not cache_path.exists():
        return {}
    try:
        with cache_path.open("r", encoding="utf-8") as f:
            cache_data = json.load(f)
        return cache_data if isinstance(cache_data, dict) else {}
    except Exception:
        return {}


def save_cache(data, cache_path=RESULTS_CACHE_PATH):
    if not USE_RESULT_CACHE:
        return
    cache_data = load_cache(cache_path)
    cache_data.update(data)
    with cache_path.open("w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)


MODEL_CONFIGS = {
    "question_raiser": "gpt-5-mini",
    "judge": "gpt-5-mini",
    "competitor_models": {
        "openai": {
            "provider": "openai",
            "model": "gpt-5-nano",
        },
        "anthropic": {
            "provider": "anthropic",
            "model": "claude-sonnet-4-5",
            "max_tokens": 1000,
        },
        "gemini": {
            "provider": "openai-compatible",
            "model": "gemini-2.5-flash",
            "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
            "api_key": google_api_key,
        },
        "deepseek": {
            "provider": "openai-compatible",
            "model": "deepseek-chat",
            "base_url": "https://api.deepseek.com/v1",
            "api_key": deepseek_api_key,
        },
        "ollama": {
            "provider": "openai-compatible",
            "model": "llama3.2",
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama",
        },
    },
}


def _anthropic_messages(prompt):
    return [{"role": "user", "content": prompt}]


def generate_answer(prompt, model_cfg):
    if isinstance(model_cfg, str):
        model_cfg = {"provider": "openai", "model": model_cfg}

    provider = model_cfg["provider"]

    if provider == "anthropic":
        client = Anthropic(api_key=anthropic_api_key)
        response = client.messages.create(
            model=model_cfg["model"],
            messages=_anthropic_messages(prompt),
            max_tokens=model_cfg.get("max_tokens", 1000),
        )
        return response.content[0].text

    client = OpenAI(
        api_key=model_cfg.get("api_key"),
        base_url=model_cfg.get("base_url"),
    )
    response = client.chat.completions.create(
        model=model_cfg["model"],
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [ ]:
# The challenging question

request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."

In [ ]:
cache_data = load_cache()
cached_question = cache_data.get("question")

# Reuse cached question so cached competitor answers stay relevant to the same prompt.
if cached_question:
    question = cached_question
    if USE_RESULT_CACHE:
        print(f"Loaded question from cache: {RESULTS_CACHE_PATH}")
else:
    question = generate_answer(request, MODEL_CONFIGS["question_raiser"])
    save_cache({"question_request": request, "question": question})
    if USE_RESULT_CACHE:
        print(f"Generated and cached question: {RESULTS_CACHE_PATH}")
    else:
        print("Generated question (cache disabled)")

if PRINT_QUESTION:
    display(Markdown(question))


In [ ]:
competitors = []
competitor_keys = []

In [ ]:

def run_all_models(
    model_configs,
    iteration_key,
    default_prompt=None,
    prompts_by_model=None,
    cache_path=RESULTS_CACHE_PATH,
):
    if prompts_by_model is None and default_prompt is None:
        raise ValueError("Provide either default_prompt or prompts_by_model")

    cache_data = load_cache(cache_path)
    results_by_iteration = cache_data.get("results_by_iteration", {})
    cached_results = results_by_iteration.get(iteration_key)
    if isinstance(cached_results, dict) and all(
        name in cached_results for name in model_configs
    ):
        return cached_results

    prompts = (
        prompts_by_model
        if prompts_by_model is not None
        else {name: default_prompt for name in model_configs}
    )

    results = {}
    with ThreadPoolExecutor(max_workers=len(model_configs)) as executor:
        futures = {
            name: executor.submit(generate_answer, prompts[name], cfg)
            for name, cfg in model_configs.items()
        }
        for name, future in futures.items():
            try:
                results[name] = future.result()
            except Exception as e:
                results[name] = f"ERROR: {e}"

    results_by_iteration[iteration_key] = results
    save_cache({"results_by_iteration": results_by_iteration}, cache_path)

    return results


def _responses_block(answers):
    together = ""
    for index, answer in enumerate(answers):
        together += f"# Response from competitor {index+1}\n\n"
        together += answer + "\n\n"
    return together


def run_judge_decision(
    question,
    competitors,
    answers,
    iteration_key,
    cache_path=RESULTS_CACHE_PATH,
):
    judge_cfg = MODEL_CONFIGS["judge"]
    judge_model_key = judge_cfg if isinstance(judge_cfg, str) else judge_cfg["model"]

    cache_data = load_cache(cache_path)
    decisions_by_model = cache_data.get("decisions_by_model", {})
    model_decisions = decisions_by_model.get(judge_model_key, {})
    cached_decision = model_decisions.get(iteration_key)
    if isinstance(cached_decision, dict) and "descision" in cached_decision:
        return cached_decision

    together = _responses_block(answers)
    judge_prompt = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank (1st is the best) and score them 1-100  in order of best to worst.
Give a score 1-100 to the quality of the answer, eg 100 is the perfect quality, it cannot be inproved, while 0 is totally unacceptable.
Also give a recommendation what needs to be added or changed or removed from the response to improve the score.
Respond with JSON, and only JSON, with the following format:
{{\"descision\": [{{\"rank\":\"1\",\"competitor_number\":3,\"score\":98, \"recommendation\":\"elaborate point 4 better,remove the contradiction...\"}}, {{\"rank\":\"2\",\"competitor_number\":2,\"score\":70, \"recommendation\":\"the last sentence does not make sense, remove it\"}}, {{\"rank\":\"3\",\"competitor_number\":1,\"score\":22, \"recommendation\":\"add a more details to the ...\"}}, ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

    decision_text = generate_answer(judge_prompt, judge_cfg)
    decision_dict = json.loads(decision_text)
    model_decisions[iteration_key] = decision_dict
    decisions_by_model[judge_model_key] = model_decisions
    save_cache({"decisions_by_model": decisions_by_model}, cache_path)

    return decision_dict


competitor_keys = list(MODEL_CONFIGS["competitor_models"].keys())
competitors = [
    MODEL_CONFIGS["competitor_models"][key]["model"]
    for key in competitor_keys
]

first_results = run_all_models(
    MODEL_CONFIGS["competitor_models"],
    iteration_key="first_iteration",
    default_prompt=question,
)
first_answers = [first_results[key] for key in competitor_keys]
first_decision_dict = run_judge_decision(
    question,
    competitors,
    first_answers,
    iteration_key="first_iteration",
)


In [ ]:
# Build per-model second-iteration prompts using the first decision recommendations

recommendations_by_key = {}
for item in first_decision_dict.get("descision", []):
    index = int(item.get("competitor_number", 0)) - 1
    if 0 <= index < len(competitor_keys):
        model_key = competitor_keys[index]
        recommendations_by_key[model_key] = item.get(
            "recommendation", "Improve your previous answer."
        )

second_prompts_by_model = {}
for model_key in competitor_keys:
    second_prompts_by_model[model_key] = f"""Original question:
{question}

Your previous answer:
{first_results[model_key]}

Judge recommendation:
{recommendations_by_key.get(model_key, 'Improve your previous answer.')}

Rewrite and improve your answer to the original question by applying the recommendation.
Return only the improved answer."""

second_results = run_all_models(
    MODEL_CONFIGS["competitor_models"],
    iteration_key="second_iteration",
    prompts_by_model=second_prompts_by_model,
)
second_answers = [second_results[key] for key in competitor_keys]
second_decision_dict = run_judge_decision(
    question,
    competitors,
    second_answers,
    iteration_key="second_iteration",
)


In [ ]:
# Compare first and second decisions: rank changes and score deltas

def decision_by_model(decision_dict, competitor_keys):
    data = {}
    for item in decision_dict.get("descision", []):
        index = int(item.get("competitor_number", 0)) - 1
        if 0 <= index < len(competitor_keys):
            model_key = competitor_keys[index]
            data[model_key] = {
                "rank": int(item.get("rank", 0)),
                "score": float(item.get("score", 0)),
            }
    return data

first_map = decision_by_model(first_decision_dict, competitor_keys)
second_map = decision_by_model(second_decision_dict, competitor_keys)

print("Comparison of first_iteration vs second_iteration")
for model_key in competitor_keys:
    model_name = MODEL_CONFIGS["competitor_models"][model_key]["model"]
    first_item = first_map.get(model_key, {"rank": 0, "score": 0.0})
    second_item = second_map.get(model_key, {"rank": 0, "score": 0.0})
    rank_changed = first_item["rank"] != second_item["rank"]
    score_delta = second_item["score"] - first_item["score"]
    print(
        f"{model_name}: rank {first_item['rank']} -> {second_item['rank']} "
        f"(changed={rank_changed}), score {first_item['score']} -> {second_item['score']} "
        f"(delta={score_delta:+.1f})"
    )

best_results = []
for model_key in competitor_keys:
    model_name = MODEL_CONFIGS["competitor_models"][model_key]["model"]
    first_score = first_map.get(model_key, {"score": 0.0})["score"]
    second_score = second_map.get(model_key, {"score": 0.0})["score"]
    if second_score > first_score:
        best_score = second_score
        best_iteration = "second_iteration"
    else:
        best_score = first_score
        best_iteration = "first_iteration"
    best_results.append((model_name, best_score, best_iteration))

best_results.sort(key=lambda x: x[1], reverse=True)

print("\nFinal ranking by best score across iterations")
for rank, (model_name, best_score, best_iteration) in enumerate(best_results, start=1):
    print(
        f"{rank}. {model_name} | best score: {best_score} | iteration: {best_iteration}"
    )

if PRINT_ANSWER:
    display(Markdown("## Answers by Model and Iteration"))
    for model_key in competitor_keys:
        model_name = MODEL_CONFIGS["competitor_models"][model_key]["model"]
        first_score = first_map.get(model_key, {"score": 0.0})["score"]
        second_score = second_map.get(model_key, {"score": 0.0})["score"]

        first_md = f"""### {model_name}
**Iteration:** first_iteration
**Judge score:** {first_score}

**Answer:**
{first_results.get(model_key, '')}
"""
        display(Markdown(first_md))

        second_md = f"""### {model_name}
**Iteration:** second_iteration
**Judge score:** {second_score}

**Answer:**
{second_results.get(model_key, '')}
"""
        display(Markdown(second_md))
